# Day 25 — Delta Lake Advanced

**What you will learn:**
- `OPTIMIZE` — compaction: merging small files into large ones
- `Z-ORDER BY` — co-locating related data for faster skipping
- `VACUUM` — removing old Parquet files to reclaim storage
- Change Data Feed (CDF) — reading only changed rows
- Delta table properties and tuning
- Liquid clustering (Databricks 13.3+)

**Exam relevance:** Delta Lake is a key exam domain — OPTIMIZE, VACUUM, and CDF appear in almost every exam.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp
import tempfile, os

DELTA_PATH = tempfile.mkdtemp(prefix="delta_adv_")
print(f"Delta path: {DELTA_PATH}")

# Seed data — many small writes to create many small Parquet files
for i in range(5):
    batch = spark.createDataFrame([
        (i * 10 + j, f"user_{i*10+j}", ["EU", "US", "APAC"][j % 3], (i * 10 + j) * 1000)
        for j in range(10)
    ], ["id", "name", "region", "revenue"])
    batch.write.format("delta").mode("append").save(DELTA_PATH)

dt = DeltaTable.forPath(spark, DELTA_PATH)
print(f"Rows: {spark.read.format('delta').load(DELTA_PATH).count()}")

## 1. OPTIMIZE — Compaction

Many small writes create many small Parquet files (the "small files problem").
- Each file requires a separate read task → too many tasks
- Metadata overhead grows

`OPTIMIZE` rewrites small files into larger ones (default target: 1GB).

> **Exam tip:** OPTIMIZE does NOT delete old files — that's VACUUM. After OPTIMIZE, the old small files still exist in the `_delta_log` but are marked as removed.

In [ ]:
# OPTIMIZE via SQL
spark.sql(f"OPTIMIZE delta.`{DELTA_PATH}`")

# Or via DeltaTable API (Databricks)
# dt.optimize().executeCompaction()

print("After OPTIMIZE: files compacted into fewer, larger Parquet files.")
print("History:")
dt.history().select("version", "operation").show()

## 2. Z-ORDER BY — Multi-Dimensional Clustering

Z-ORDER co-locates related rows within the same Parquet file.
- Min/max statistics per file become tighter
- Delta's data skipping skips more files on queries that filter by those columns

```sql
OPTIMIZE delta.`path` ZORDER BY (region, revenue)
```

**When Z-ORDER helps:**
- High-cardinality columns used in WHERE filters
- Frequently joined columns

> **Exam tip:** Z-ORDER is effective for read performance but adds write cost. It's not a replacement for partitioning — combine both for best results.

In [ ]:
# OPTIMIZE with Z-ORDER
spark.sql(f"OPTIMIZE delta.`{DELTA_PATH}` ZORDER BY (region, revenue)")
print("Z-ORDER OPTIMIZE complete.")

# A query filtering by region now benefits from data skipping
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("region") == "EU") \
    .explain("formatted")

## 3. VACUUM — Removing Obsolete Files

Delta tables accumulate old Parquet files that are no longer part of the current table state (after OPTIMIZE, UPDATE, DELETE).

`VACUUM` deletes files older than the **retention threshold** (default: 7 days).

> **Exam tip:** After VACUUM, time travel to versions older than the retention window is no longer possible — the files are gone. VACUUM is irreversible.

In [ ]:
# VACUUM — default 168 hours (7 days) retention
# In production: spark.sql(f"VACUUM delta.`{DELTA_PATH}`")

# Dry run first — lists what would be deleted
spark.sql(f"VACUUM delta.`{DELTA_PATH}` RETAIN 0 HOURS DRY RUN").show(truncate=False)

# DANGER: 0-hour retention deletes ALL old files — disables time travel
# Only do this in dev/testing. Requires disabling safety check:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM delta.`{DELTA_PATH}` RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("VACUUM complete.")

## 4. Change Data Feed (CDF)

CDF lets you read only the **rows that changed** between two versions.

Each changed row has an extra column `_change_type`:
- `insert` — row was inserted
- `update_preimage` — row value BEFORE the update
- `update_postimage` — row value AFTER the update
- `delete` — row was deleted

In [ ]:
# Create a fresh Delta table with CDF enabled
CDF_PATH = tempfile.mkdtemp(prefix="delta_cdf_")

spark.createDataFrame([
    (1, "Alice", 95000),
    (2, "Bob",   72000),
    (3, "Carol", 110000),
], ["id", "name", "salary"]) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(CDF_PATH)

dt_cdf = DeltaTable.forPath(spark, CDF_PATH)

# Make some changes — version 1
dt_cdf.update(col("id") == 1, {"salary": lit(100000)})
dt_cdf.delete(col("id") == 2)

spark.createDataFrame([(4, "Dave", 80000)], ["id", "name", "salary"]) \
    .write.format("delta").mode("append").save(CDF_PATH)

# Read CDF — all changes from version 1 onwards
spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .load(CDF_PATH) \
    .orderBy("id", "_commit_version") \
    .show()

## 5. Delta Table Properties

In [ ]:
# Set table properties
spark.sql(f"""
    ALTER TABLE delta.`{DELTA_PATH}`
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true',
        'delta.dataSkippingNumIndexedCols' = '4'
    )
""")

# Inspect table properties and file stats
spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_PATH}`").show(vertical=True)

## 6. Summary: Delta Maintenance Operations

| Operation | What it does | Removes data? |
|---|---|---|
| `OPTIMIZE` | Compacts small files into larger ones | No |
| `OPTIMIZE ZORDER BY` | Compacts + co-locates data by columns | No |
| `VACUUM` | Deletes files outside retention window | **Yes** — irreversible |
| `DESCRIBE HISTORY` | Shows all commits | No |
| `RESTORE` | Restores table to a previous version | No (creates new version) |

---

## Day 25 Checklist

- [ ] Know what OPTIMIZE does (compact) and what it does NOT do (delete)
- [ ] Know Z-ORDER: co-locates data → tighter min/max stats → better data skipping
- [ ] Know VACUUM default retention (7 days / 168 hours) — irreversible
- [ ] Enabled CDF with `delta.enableChangeDataFeed` property
- [ ] Read CDF with `readChangeFeed=true` + `startingVersion`
- [ ] Know CDF change types: insert, update_preimage, update_postimage, delete

**Next:** Day 26 — Spark Connect & Unity Catalog